In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
from scipy.special import expit
#from missmecha.generator import MissMechaGenerator
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import *

In [2]:
class auxilliary_methods:
    @staticmethod
    def add_intercept_column(df):
        df.insert(0, 'intercept', 1)
        return df
    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    @staticmethod
    def logistic_missingness_loss(xc,bias,steepness,missing_proba):
        """
        Method used for getting the right bias so that missing probability is as provided in missing_proba argument for observations from provided dataset. Takes arguments xc - which corresponds either to selected column in MAR1 setting or whole X scaled with weights in MAR2 setting, steepness parameter of sigmoid and bias - the proposed bias value to be evaluated.
        """
        proba = expit(steepness * (xc - bias))
        return (proba.mean() - missing_proba) ** 2

In [3]:
class missing_value_generator:
    @staticmethod
    def MCAR(X,Y,missing_proba,seed=None):
        """
        Function to prepare data with missing values using missing completly at random approach. Takes X - features, Y - target and missing_proba - the percentage of values to be made missing as arguments. Observations with missing label selected using variables from uniform distribution on [0,1]. Returns a vector new_y with some of the values of Y made missing. Seed may be set for reproducibility.
        """
        if seed is not None:
            np.random.seed(seed)
        l=len(Y)
        new_y=np.zeros(l)
        make_missing=np.random.uniform(0,1,l)
        for j in range(l):
            if make_missing[j]<missing_proba:
                new_y[j]=-1
            else:
                new_y[j]=Y[j]
        return new_y
    @staticmethod
    def MAR1(X,Y,missing_proba,missing_influence_col_index,steepness=1.0,seed=None):
        """
        Function to prepare data with missing values using missing at random approach depending on one column passed as argument. Takes X - features, Y - target, missing_influence_col_index - position of column that influences the missingness probability and missing_proba - the percentage of values to be made missing as arguments. Observations with missing label selected using from uniform distribution on [0,1], but for each row the threshold is set to different value depending on the value of column at position missing_influence_column_index . Returns a vector new_y with some of the values of Y made missing. We assume that at this point data is standardized. Seed may be set for reproducibility.
        """
        if missing_influence_col_index<0 or missing_influence_col_index>=X.shape[1]:
            raise ValueError(f"Wrong column index. Index should be between 0 and {X.shape[1]-1}")
        if seed is not None:
            np.random.seed(seed)
        l=len(Y)
        new_y=np.zeros(l)
        xc=X[:,missing_influence_col_index]
        result = minimize_scalar(lambda bias: auxilliary_methods.logistic_missingness_loss(xc, bias, steepness, missing_proba), bounds=(-10, 10), method='bounded')
        bias = result.x
        per_row_probabilities=expit(steepness * (xc-bias))
        make_missing=np.random.uniform(0,1,l)
        for j in range(l):
            if make_missing[j]<per_row_probabilities[j]:
                new_y[j]=-1
            else:
                new_y[j]=Y[j]
        return new_y
    @staticmethod
    def MAR2(X,Y,missing_proba,steepness=1.0,weights=None,seed=None):
        """
        Function to prepare data with missing values using missing at random approach depending on all columns. Takes X - features, Y - target missing_proba - the percentage of values to be made missing, steepness - parameter influencing how the probabilities change with values and weights of columns - how they affect the missing probability on a single observation, the last two being optional parameters as arguments. Observations with missing label selected using from uniform distribution on [0,1], but for each row the threshold is set to different value depending on the value of column at position missing_influence_column_index . Returns a vector new_y with some of the values of Y made missing. We assume that at this point data is standardized. Seed may be set for reproducibility.
        """
        if seed is not None:
            np.random.seed(seed)
        n_cols = X.shape[1]
        l=len(Y)
        if weights is None:
            weights_used = np.ones(n_cols) / np.sqrt(n_cols)
        else:
            weights_used = np.array(weights)/np.linalg.norm(weights)
        xc=X@weights_used
        result = minimize_scalar(lambda bias: auxilliary_methods.logistic_missingness_loss(xc, bias, steepness, missing_proba), bounds=(-10, 10), method='bounded')
        bias = result.x
        per_row_probabilities=expit(steepness * (xc-bias))
        make_missing=np.random.uniform(0,1,l)
        new_y=np.zeros(l)
        for j in range(l):
            if make_missing[j]<per_row_probabilities[j]:
                new_y[j]=-1
            else:
                new_y[j]=Y[j]
        return new_y
    @staticmethod
    def MNAR(X,Y,missing_proba,seed=None):
        if seed is not None:
            np.random.seed(seed)
        l=len(Y)
        new_y=np.zeros(l)
        lr=LogisticRegression()
        lr.fit(X,Y)
        probabilities=lr.predict_proba(X)[:, 1]
        unnormalizedmissingprobas=0.5*probabilities+0.5*Y
        m=np.mean(unnormalizedmissingprobas)
        normalizedmissingprobas=unnormalizedmissingprobas*missing_proba/m.astype(float)
        make_missing=np.random.uniform(0,1,l)
        for j in range(l):
            if normalizedmissingprobas[j]>1:
                normalizedmissingprobas[j]=1
            if make_missing[j]<normalizedmissingprobas[j]:
                new_y[j]=-1
            else:
                new_y[j]=Y[j]
        return new_y











In [76]:
class LassoAuxiliary:
    @staticmethod
    def lasso_penalty(w,llambda):
        return llambda*np.linalg.norm(w[1:],ord=1)#assumes intercept is present
    @staticmethod
    def logistic_loss(w, X, y):
        p = 1 / (1 + np.exp(-X @ w))
        return -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))

    @staticmethod
    def logistic_loss_gradient(w, X, y):
        p = 1 / (1 + np.exp(-X @ w))
        return X.T @ (p - y) / len(y)
    @staticmethod
    def lipschitzvalue(A):
        n = A.shape[0]
        L = (1 / (4 * n)) * (np.linalg.norm(A, ord=2) ** 2)
        return L
    @staticmethod
    def soft_thresholding_operator_for_lasso(w,l):
        return np.sign(w)*np.maximum(np.abs(w)-l,0)
    @staticmethod
    def fista_lasso_solver(A,y,llambda,n_iter=100):
        """
        params: A -experiment matrix,y-training labels, llambda - penalty coefficient, n_iter - number of iterations
        """
        #initialization part
        n_features=A.shape[1]
        L=LassoAuxiliary.lipschitzvalue(A)
        s=llambda/L
        x=np.zeros(n_features)
        x_new=np.zeros(n_features)
        z=np.zeros(n_features)
        t=1
        t_new=1
        for i in range(n_iter):
            x_new=LassoAuxiliary.soft_thresholding_operator_for_lasso(z-(1/L)*LassoAuxiliary.logistic_loss_gradient(z,A,y),s)
            t_new = (1+np.sqrt(1+4*t**2))/2
            z_new = x_new + ((t-1)/(t_new))*(x_new - x)
            x = x_new
            z = z_new
            t = t_new
        return x

In [34]:
class FistaLogisticLassoRegressionClassifierFamily:
    def __init__(self,threshold=0.5,random_state=2137,lambdas=None):
        self.threshold=threshold
        self.random_state=random_state
        if lambdas is None:
            self.lambdas=np.logspace(-3,1,10)
        else:
            self.lambdas=lambdas
        self.best_lambda=None
        self.weights={key:[] for key in self.lambdas}
    def validate(self,X_validate,y_validate,lambda_,measure='accuracy'):
        w=self.weights[lambda_]
        z = X_validate @ w
        probas=auxilliary_methods.sigmoid(z)
        predictions=probas>0.5
        if measure=='accuracy':
            return accuracy_score(y_validate,predictions)
        elif measure=='precision':
            return precision_score(y_validate,predictions)
        elif measure=='recall':
            return recall_score(y_validate,predictions)
        elif measure=='f1':
            return f1_score(y_validate,predictions)
        elif measure=='auc':
            return roc_auc_score(y_validate,predictions)
        elif measure=='balanced_accuracy':
            return balanced_accuracy_score(y_validate,predictions)
        elif measure=='spc':
            return average_precision_score(y_validate,predictions)
        else:
            raise ValueError(f"Invalid measure: {measure}. Valid measures are: accuracy, precision, recall, auc, f1, balanced_accuracy, spc")
    def fit(self,X_train,y_train):
        best_result=0
        best_lambda=self.lambdas[0]
        for lambda_ in self.lambdas:
            self.weights[lambda_]=LassoAuxiliary.fista_lasso_solver(X_train,y_train,lambda_)
            valres=self.validate(X_train,y_train,lambda_)
            if valres>best_result:
                best_lambda=lambda_
                best_result=valres
        self.best_lambda=best_lambda
    def predict_proba(self,X):
        w=self.weights[self.best_lambda]
        z = X @ w
        return auxilliary_methods.sigmoid(z)
    def plot(self,measure,X_validate,y_validate,lambdas=None):
        if lambdas is None:
            lambdas=self.lambdas
        values=np.zeros(len(lambdas))
        for i in range(len(lambdas)):
            values[i]=self.validate(X_validate,y_validate,lambdas[i],measure)
        plt.plot(lambdas,values)
        plt.title(f"Influence of lambda on model's {measure}.")
        plt.xlabel("lambda")
        plt.ylabel(f"{measure}")
        plt.show()
    def plot_coefficients(self):
        for i in range(len(self.weights[self.best_lambda])):
            coefs=[]
            for lambda_ in self.lambdas:
                coefs.append(self.weights[lambda_][i])
            plt.scatter(self.lambdas,coefs)
            plt.xlabel("lambda")
            plt.ylabel(f"coefficient {i}")
            plt.show()



In [6]:
class UnlabeledLogReg:
    def __init__(self,imputation_type):
        if imputation_type not in ["logistic","rf","tree","knn","prior"]:
            raise ValueError(f"Invalid imputation type: {imputation_type}")
        self.imputation_type=imputation_type
    def partial_model_imputation(self, X, Y, modeltype="logistic"):
        """
        Method for imputation of missing data using a model of the type specified by modeltype parameter trained on the non-missing subset of data. Available model types are: logistic regression (default value), random forest and single decision tree. If other string is passed to modeltype parameter, default model type will be used, but the user will be informed.
        """
        Y = np.array(Y)
        mask = (Y != -1)
        X_mod = X.iloc[mask, :]
        Y_mod = Y[mask]
        X_miss = X.iloc[~mask, :]
        if modeltype == "logistic":
            model = LogisticRegression()
        elif modeltype == "rf":
            model = RandomForestClassifier(n_estimators=10, random_state=0, max_depth=5)
        elif modeltype == "tree":
            model = DecisionTreeClassifier(random_state=0, max_depth=5)
        else:
            print("Unknown model type! Switching to default: logistic regression")
            model = LogisticRegression()

        model.fit(X_mod, Y_mod)
        y_imputed = model.predict(X_miss)
        Y[~mask] = y_imputed
        return Y

    def KNN_imputation(self, X, Y, neighbor_count=5):
        Y = np.array(Y)
        mask = (Y != -1)

        X_mod = X.iloc[mask, :]
        Y_mod = Y[mask]
        X_miss = X.iloc[~mask, :]

        knn = KNeighborsClassifier(n_neighbors=neighbor_count)
        knn.fit(X_mod, Y_mod)

        Y[~mask] = knn.predict(X_miss)
        return Y

    def prior_probability_imputation(self, Y):
        """
        Oversimplifying imputation that calculates class prior in non-missing subset and then imputes missing values at random using bernoulli distribution with prior probability of success.
        """
        Y = np.array(Y)
        mask = (Y != -1)

        Y_mod = Y[mask]
        positive_prior = np.mean(Y_mod)
        miss_ct = np.sum(~mask)
        Y[~mask] = np.random.binomial(1, positive_prior, miss_ct)
        return Y
    def fit(self,X,Y):
        if self.imputation_type == "knn":
            new_y=self.KNN_imputation(X,Y)
        elif self.imputation_type == "prior":
            new_y=self.prior_probability_imputation(Y)
        else:
            new_y=self.partial_model_imputation(X,Y,self.imputation_type)
        clasifier=FistaLogisticLassoRegressionClassifierFamily()
        clasifier.fit(X,Y)
        self.clasifier=clasifier

